In [1]:
import pandas as pd
import urllib.request
import zipfile
import os

# Автоматичне завантаження та розпакування датасету
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00235/household_power_consumption.zip"
zip_path = "../household_power_consumption.zip"
extract_dir = "../power_data"

if not os.path.exists(extract_dir):
    os.makedirs(extract_dir)
    print("Завантажуємо датасет..")
    urllib.request.urlretrieve(url, zip_path)
    print("Розпаковуємо архів...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print("Готово!")
else:
    print("Датасет вже завантажено та розпаковано.")

# Зчитування та Data Cleaning
file_path = os.path.join(extract_dir, "household_power_consumption.txt")

print("Зчитуємо та чистимо дані///")
# Зчитуємо дані. Розділювач тут - крапка з комою. Пропуски у цьому датасеті позначені як '?'
# Відразу вказуємо na_values=['?'], щоб Pandas сприймав їх як NaN
df_power = pd.read_csv(file_path, sep=';', na_values=['?'], low_memory=False)

# Видаляємо всі рядки з пропущеними даними
df_power = df_power.dropna()

print(f"Дані успішно очищено! Залишилося рядків: {len(df_power)}")
df_power.head()

Завантажуємо датасет..
Розпаковуємо архів...
Готово!
Зчитуємо та чистимо дані///
Дані успішно очищено! Залишилося рядків: 2049280


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


In [2]:

def filter_active_power(df):
    """
    Фільтрує датафрейм, залишаючи записи з Global_active_power > 5 кВт
    """
    # Global_active_power у нас вже числова колонка після очищення
    return df[df['Global_active_power'] > 5.0]

print("Вимірюємо час виконання функції за допомогою timeit...")
#  команда для профілювання часу
%timeit filter_active_power(df_power)

# Зберігаємо результат у змінну, щоб подивитися на дані
high_power_df = filter_active_power(df_power)

print(f"\nКількість знайдених записів: {len(high_power_df)}")
high_power_df.head()

Вимірюємо час виконання функції за допомогою timeit...
6.01 ms ± 1.01 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)

Кількість знайдених записів: 17547


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
11,16/12/2006,17:35:00,5.412,0.470,232.78,23.2,0.0,1.0,17.0
12,16/12/2006,17:36:00,5.224,0.478,232.99,22.4,0.0,1.0,16.0


In [3]:
def filter_intensity_and_appliances(df):
    """
    Фільтрує за силою струму (19-20 А) та порівнює споживання груп приладів
    Згідно з документацією датасету:
    Sub_metering_2 - це пральна машина, холодильник тощо
    Sub_metering_3 - це бойлер та кондиціонер
    """
    # Сила струму від 19 до 20 А включно
    intensity_mask = (df['Global_intensity'] >= 19) & (df['Global_intensity'] <= 20)

    # Пралка та холодильник > бойлер та кондиціонер
    appliance_mask = df['Sub_metering_2'] > df['Sub_metering_3']

    # Об'єднуємо умови та фільтруємо
    return df[intensity_mask & appliance_mask]

print("Вимірюємо час виконання за допомогою timeit...")
%timeit filter_intensity_and_appliances(df_power)

filtered_appliances_df = filter_intensity_and_appliances(df_power)
print(f"\nКількість знайдених записів: {len(filtered_appliances_df)}")
filtered_appliances_df.head()

Вимірюємо час виконання за допомогою timeit...
18.8 ms ± 4.52 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)

Кількість знайдених записів: 2509


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
45,16/12/2006,18:09:00,4.464,0.136,234.66,19.0,0.0,37.0,16.0
460,17/12/2006,01:04:00,4.582,0.258,238.08,19.6,0.0,13.0,0.0
464,17/12/2006,01:08:00,4.618,0.104,239.61,19.6,0.0,27.0,0.0
475,17/12/2006,01:19:00,4.636,0.140,237.37,19.4,0.0,36.0,0.0
476,17/12/2006,01:20:00,4.634,0.152,237.17,19.4,0.0,35.0,0.0


In [4]:
def random_sample_and_mean(df):
    """
    Обирає випадковим чином 500 000 записів (без повторів)
    та обчислює середні величини для Sub_metering_1, 2 та 3.
    """
    # Відбираємо 500 000 випадкових записів.
    # replace=False гарантує, що елементи не будуть повторюватися.
    # random_state=42 фіксує "випадковість", щоб результати були однаковими при кожному запуску (опціонально)
    sampled_df = df.sample(n=500000, replace=False, random_state=42)

    # Обчислюємо середнє (mean) для потрібних колонок
    means = sampled_df[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()

    return means

print("Вимірюємо час виконання за допомогою timeit...")
%timeit random_sample_and_mean(df_power)

# Зберігаємо та виводимо результат
means_result = random_sample_and_mean(df_power)
print("\nСередні величини споживання для 3-х груп (вибірка 500,000 записів):")
print(means_result)

Вимірюємо час виконання за допомогою timeit...
329 ms ± 139 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)

Середні величини споживання для 3-х груп (вибірка 500,000 записів):
Sub_metering_1    1.119258
Sub_metering_2    1.308912
Sub_metering_3    6.452950
dtype: float64


In [5]:
def complex_slicing(df):

    # Час після 18:00 (оскільки формат HH:MM:SS, рядкове порівняння працює ідеально)
    time_mask = df['Time'] >= '18:00:00'

    # Споживають понад 6 кВт за хвилину (Global_active_power - це хвилинне споживання)
    power_mask = df['Global_active_power'] > 6.0

    # Основне споживання - це група 2 (Sub_metering_2 більша за 1 та 3)
    group2_mask = (df['Sub_metering_2'] > df['Sub_metering_1']) & (df['Sub_metering_2'] > df['Sub_metering_3'])

    # Об'єднуємо всі маски та відкидаємо старі індекси
    filtered_df = df[time_mask & power_mask & group2_mask].reset_index(drop=True)

    # Ділимо датасет на дві половини
    half_index = len(filtered_df) // 2
    first_half = filtered_df.iloc[:half_index]
    second_half = filtered_df.iloc[half_index:]

    # Обираємо кожен 3-й запис з першої половини та кожен 4-й з другої
    # Синтаксис [::3] означає , що потрібно брати кожен третій крок"
    result_first = first_half.iloc[::3]
    result_second = second_half.iloc[::4]

    # Зліплюємо результати назад в одну таблицю
    final_result = pd.concat([result_first, result_second])

    return final_result

print("Вимірюємо час виконання за допомогою timeit...")
%timeit complex_slicing(df_power)

final_complex_df = complex_slicing(df_power)
print(f"\nКількість знайдених записів після всіх фільтрів та зрізів: {len(final_complex_df)}")
final_complex_df.head()

Вимірюємо час виконання за допомогою timeit...
160 ms ± 6.06 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)

Кількість знайдених записів після всіх фільтрів та зрізів: 310


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,18:05:00,6.052,0.192,232.93,26.2,0.0,37.0,17.0
3,16/12/2006,18:08:00,6.308,0.116,232.25,27.0,0.0,36.0,17.0
6,28/12/2006,20:58:00,6.386,0.374,236.63,27.0,1.0,36.0,17.0
9,28/12/2006,21:02:00,8.088,0.262,235.50,34.4,1.0,72.0,17.0
12,28/12/2006,21:05:00,7.230,0.152,235.22,30.6,1.0,73.0,17.0


In [7]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Нормалізація та стандартизація
print("Нормалізація та стандартизація")
# Візьмемо невеликий семпл, щоб не вантажити пам'ять, і оберемо числові колонки
sample_df = df_power[['Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity']].sample(1000, random_state=42)

# Стандартизація (Z-score normalization, середнє=0, дисперсія=1)
scaler_standard = StandardScaler()
df_standardized = pd.DataFrame(scaler_standard.fit_transform(sample_df), columns=sample_df.columns)
print("Стандартизовані дані (перші 3 рядки):")
display(df_standardized.head(3))

# Нормалізація (MinMax scaling, значення від 0 до 1)
scaler_minmax = MinMaxScaler()
df_normalized = pd.DataFrame(scaler_minmax.fit_transform(sample_df), columns=sample_df.columns)
print("\nНормалізовані дані (перші 3 рядки):")
display(df_normalized.head(3))

# Кореляція Пірсона та Спірмена
print("\nКореляція Пірсона та Спірмена")
# Рахуємо кореляцію між aктивною потужністю та cилою струму
col1 = 'Global_active_power'
col2 = 'Global_intensity'

pearson_corr = sample_df[col1].corr(sample_df[col2], method='pearson')
spearman_corr = sample_df[col1].corr(sample_df[col2], method='spearman')

print(f"Коефіцієнт Пірсона між {col1} та {col2}: {pearson_corr:.4f}")
print(f"Коефіцієнт Спірмена між {col1} та {col2}: {spearman_corr:.4f}")
print("Значення близькі до 1, що логічно, бо чим більший струм -- тим більша потужність")

# One Hot Encoding
print("\nOne Hot Encoding")
# У нашому датасеті немає явних категоріальних атрибутів (типу "Колір" або "Місто").
# Але ми можемо витягнути місяць з дати і зробити ohe для нього.
sample_df_with_date = df_power[['Date', 'Global_active_power']].sample(10, random_state=42).copy()

# Витягуємо місяць (DD/MM/YYYY)
sample_df_with_date['Month'] = sample_df_with_date['Date'].str.split('/').str[1]

# Робимо One Hot Encoding (pd.get_dummies)
ohe_df = pd.get_dummies(sample_df_with_date, columns=['Month'], prefix='Month')
print("Результат One Hot Encoding для колонки 'Month':")
display(ohe_df)

Нормалізація та стандартизація
Стандартизовані дані (перші 3 рядки):


,Global_active_power,Global_reactive_power,Voltage,Global_intensity
0,0.442370,-0.426333,-0.237551,0.452415
1,-0.674092,1.217135,1.444992,-0.629355
2,-0.430608,1.528529,-0.338567,-0.347154



Нормалізовані дані (перші 3 рядки):


,Global_active_power,Global_reactive_power,Voltage,Global_intensity
0,0.208872,0.082960,0.529095,0.206667
1,0.043184,0.295964,0.789731,0.053333
2,0.079318,0.336323,0.513447,0.093333



Кореляція Пірсона та Спірмена
Коефіцієнт Пірсона між Global_active_power та Global_intensity: 0.9987
Коефіцієнт Спірмена між Global_active_power та Global_intensity: 0.9945
Значення близькі до 1, що логічно, бо чим більший струм -- тим більша потужність

One Hot Encoding
Результат One Hot Encoding для колонки 'Month':


,Date,Global_active_power,Month_1,Month_12,Month_5,Month_6,Month_8
1030580,1/12/2008,1.502,False,True,False,False,False
1815,17/12/2006,0.374,False,True,False,False,False
1295977,3/6/2009,0.620,False,False,False,True,False
206669,9/5/2007,0.280,False,False,True,False,False
1048893,14/12/2008,1.372,False,True,False,False,False
270672,22/6/2007,0.284,False,False,False,True,False
1395384,11/8/2009,0.196,False,False,False,False,True
735922,10/5/2008,2.234,False,False,True,False,False
1566601,8/12/2009,0.486,False,True,False,False,False
1605907,4/1/2010,1.842,True,False,False,False,False
